In [10]:
!pip install xgboost shap streamlit pyngrok joblib -q

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_curve, auc
import shap
import joblib

# Load uploaded dataset
df = pd.read_csv('creditcard.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Fraud Rate: {(df['Class'].sum() / len(df)) * 100:.4f}%")

Dataset Shape: (284807, 31)
Fraud Rate: 0.1727%


In [11]:
# 1. Scale Time and Amount
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# 2. Stratified Train-Test Split
X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. XGBoost with Cost-Weighted Imbalance Handling
ratio = (len(y_train) - sum(y_train)) / sum(y_train)
model = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42
)
model.fit(X_train, y_train)

# 4. Evaluate PR-AUC Score
y_pred_proba = model.predict_proba(X_test)[:, 1]
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)
print(f"Model Training Complete. PR-AUC Score: {pr_auc:.4f}")

# 5. Export Artifacts for Streamlit App
joblib.dump(model, 'xgb_fraud_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
X_test.head(100).to_csv('sample_transactions.csv', index=False)
print("Artifacts successfully saved!")

Model Training Complete. PR-AUC Score: 0.8383
Artifacts successfully saved!


In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import joblib
import matplotlib.pyplot as plt

st.set_page_config(page_title="Payment Risk Engine", layout="wide")

st.title("💳 Real-Time Payment Risk & Anomaly Scoring Engine")
st.markdown("Simulated live transaction triage, cost-loss optimization & explainable AI risk scoring.")

@st.cache_resource
def load_assets():
    model = joblib.load('xgb_fraud_model.pkl')
    samples = pd.read_csv('sample_transactions.csv')
    explainer = shap.TreeExplainer(model)
    return model, samples, explainer

model, samples, explainer = load_assets()

# Sidebar Controls
st.sidebar.header("1. Inspector Settings")
options = [f"Transaction #{i}" for i in range(len(samples))]
selected_option = st.sidebar.selectbox("Select Sample Transaction", options, index=5)
tx_index = int(selected_option.split("#")[1])

st.sidebar.divider()
st.sidebar.header("2. Risk Appetite & Loss Matrix")
threshold = st.sidebar.slider("Decision Cutoff Threshold (%)", min_value=1.0, max_value=99.0, value=35.0, step=1.0) / 100.0
friction_cost = st.sidebar.number_input("Compliance Review Cost ($/review)", value=15.0)

sample_tx = samples.iloc[[tx_index]]

# Live Model Scoring
prob = model.predict_proba(sample_tx)[0][1]
risk_score = prob * 100

# Unscale dollar amount estimate for realistic financial metrics
raw_amount = float(np.abs(sample_tx['scaled_amount'].values[0]) * 120 + 25)

# Financial Loss Matrix Calculations
expected_fraud_loss = prob * raw_amount

st.subheader(f"Transaction Reference: TXN-{tx_index + 1000}")

col1, col2, col3, col4 = st.columns(4)
col1.metric("Est. Transaction Value", f"${raw_amount:.2f}")
col2.metric("Fraud Risk Score", f"{risk_score:.2f}%")
col3.metric("Expected Fraud Loss", f"${expected_fraud_loss:.2f}")

if prob >= threshold:
    col4.error("🚨 FLAGGED FOR COMPLIANCE REVIEW")
else:
    col4.success("✅ APPROVED")

st.divider()

# Expected Value Analysis Section
st.subheader("📊 Financial Cost-Benefit Triage Analysis")
c1, c2 = st.columns(2)
with c1:
    st.info(f"**Target Threshold Selected:** `{threshold * 100:.1f}%`\n\nIf flagged, holding this transaction costs **${friction_cost:.2f}** in operational review friction but protects **${raw_amount:.2f}** in gross exposure.")
with c2:
    if prob >= threshold:
        st.warning(f"**Action Plan:** Auto-route to Tier-2 Fraud Triage desk. Risk exposure probability exceeds corporate risk appetite cutoff of {threshold * 100:.1f}%.")
    else:
        st.success(f"**Action Plan:** Auto-cleared for immediate settlement. Risk profile sits within normal operating bounds.")

st.divider()

# Explainable AI Plot
st.subheader("Explainable AI (XAI): Feature Drivers for Risk Score")
shap_values = explainer(sample_tx)

fig, ax = plt.subplots(figsize=(8, 4))
shap.plots.waterfall(shap_values[0], max_display=7, show=False)
st.pyplot(fig)

Overwriting app.py


In [13]:
!curl ipv4.icanhazip.com

34.138.57.171


In [ ]:
# 1. Install Cloudflare tunnel
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 2. Kill old background processes
!pkill -f streamlit
!pkill -f cloudflared

# 3. Launch Streamlit
!nohup streamlit run app.py --server.enableCORS=false --server.enableXsrfProtection=false &

# 4. Open zero-config Cloudflare Tunnel
!./cloudflared tunnel --url http://localhost:8501

nohup: appending output to 'nohup.out'
2026-10-17T21:00:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-17T21:00:03Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-17T21:00:11Z INF +--------------------------------------------------------------------------------------------+
2026-10-17T21:00:11Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-17T21:00:11Z INF |  https://rolls-